In [11]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("data/chase_banking.pdf")

documents=loader.load()
print(documents)

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-03T09:29:06-04:00', 'author': 'JPMorgan Chase Bank', 'keywords': 'Chase; total; checking; guide to your account; ada; (PDF)', 'moddate': '2024-10-07T09:59:35-04:00', 'subject': 'Chase Total Checking - A Guide To Your Account', 'title': 'Chase Total Checking - A Guide To Your Account (PDF)', 'trapped': '/Unknown', 'source': 'data/chase_banking.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT\n1\nCHASE TOTAL CHECKING\n®\nA GUIDE TO YOUR ACCOUNT †\nIt’s important that you understand how your Chase Total Checking account works. \nWe’ve created this Guide to explain the fees and some key terms of your personal account.\nMONTHLY \nSERVICE FEE*\nMonthly Service Fee $12\nWays to Avoid the \nMonthly Service Fee\n$0 Monthl

In [12]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

In [13]:
from langchain_community.vectorstores import FAISS

In [14]:
faiass_db=FAISS.from_documents(documents,embeddings_model)

In [15]:
retriever=faiass_db.as_retriever(search_kwargs={"k":10})

In [17]:
response=retriever.invoke("how to open a savings accont")

In [18]:
for doc in response:
    
    print(f"Content {doc.page_content}, Metadata:{doc.metadata}")

Content HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT
1
CHASE TOTAL CHECKING
®
A GUIDE TO YOUR ACCOUNT †
It’s important that you understand how your Chase Total Checking account works. 
We’ve created this Guide to explain the fees and some key terms of your personal account.
MONTHLY 
SERVICE FEE*
Monthly Service Fee $12
Ways to Avoid the 
Monthly Service Fee
$0 Monthly Service Fee when you have any ONE  of the following during each 
monthly statement period:
•  Electronic deposits made into this account totaling $500 or more, such as 
payments from payroll providers or government benefit providers, by using 
(i) the ACH network, (ii) the Real Time Payment or FedNow SM network, or  
(ii
i) third party services that facilitate payments to your debit card using the 
Visa® or Mastercard® network
• OR, a balance at the beginning of each day of $1,500 or more in this account
•  OR, an average beginning day bal

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()
cohere_api_key=os.getenv("COHERE_API_KEY")
#print(cohere_api_key)

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

reranker=CohereRerank(model="rerank-v3.5",top_n=2)

rerank_retriever=ContextualCompressionRetriever(
    
    base_compressor=reranker,
    base_retriever=retriever
)

In [24]:
rerank_response=rerank_retriever.invoke("how to open a savings accont")

In [25]:
for doc in rerank_response:
    
    print(f"Content {doc.page_content}, Metadata:{doc.metadata}")

Content HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT
1
CHASE TOTAL CHECKING
®
A GUIDE TO YOUR ACCOUNT †
It’s important that you understand how your Chase Total Checking account works. 
We’ve created this Guide to explain the fees and some key terms of your personal account.
MONTHLY 
SERVICE FEE*
Monthly Service Fee $12
Ways to Avoid the 
Monthly Service Fee
$0 Monthly Service Fee when you have any ONE  of the following during each 
monthly statement period:
•  Electronic deposits made into this account totaling $500 or more, such as 
payments from payroll providers or government benefit providers, by using 
(i) the ACH network, (ii) the Real Time Payment or FedNow SM network, or  
(ii
i) third party services that facilitate payments to your debit card using the 
Visa® or Mastercard® network
• OR, a balance at the beginning of each day of $1,500 or more in this account
•  OR, an average beginning day bal